# APTOS 2019 Diabetic Retinopathy Grading: Colab T4 Training Notebook

This notebook is the GPU training companion for the production repo. It covers the complete pipeline requested:

1. Project setup and dependency install
2. Kaggle dataset download
3. EDA and data quality checks
4. Ben Graham / CLAHE preprocessing
5. Stratified 5-fold setup and dataloaders
6. Three model approaches: classifier, regression, ordinal/GeM
7. Training loop with AMP, QWK, checkpointing, W&B
8. Inference, TTA, fold/model ensembling, threshold optimization
9. XAI entry points: Grad-CAM and clinical explanation workflow
10. Packaging outputs for API, Streamlit, and deployment

Recommended Colab runtime: **T4 GPU**, High-RAM if available. On the menu: `Runtime -> Change runtime type -> T4 GPU`.

Clinical note: this project is for research/prototyping unless clinically validated under the appropriate regulatory pathway.

## 0. GPU Check

A T4 is enough for EfficientNet-B4/B5 at 512px and small-batch 768px training. ConvNeXt-Large and EfficientNet-B6 may require smaller batches, gradient accumulation, or Colab Pro/A100.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
print("device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Bring The Repo Into Colab

Use one of these options:

- Option A: upload this project folder as a zip to Colab and unzip to `/content/diabetes`.
- Option B: push the repo to GitHub, then clone it here.
- Option C: store the repo in Google Drive and copy/sync it into `/content`.

The notebook expects the repo root to be `/content/diabetes`.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/diabetes")

# Option B example once you push the repo:
# !git clone https://github.com/YOUR_USERNAME/diabetes.git /content/diabetes

# Option A helper: if you upload diabetes.zip through the Colab sidebar, uncomment:
# !unzip -q /content/diabetes.zip -d /content

assert PROJECT_DIR.exists(), "Put the repo at /content/diabetes first."
os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

## 2. Install Dependencies

Colab already includes many scientific packages, but pin the core stack for reproducibility. `grad-cam` is the PyPI package name; the import is `pytorch_grad_cam`.

If install asks to restart runtime after dependency changes, restart and rerun from this point.

In [ ]:
!pip install -q --upgrade pip
!pip install -q   albumentations==1.4.21   timm==1.0.12   opencv-python-headless==4.10.0.84   imagehash==4.3.1   scikit-learn==1.6.0   scikit-image==0.24.0   scipy==1.14.1   pandas==2.2.3   matplotlib==3.9.3   PyYAML==6.0.2   tqdm==4.67.1   wandb==0.19.1   grad-cam==1.5.4   lime==0.2.0.1   shap==0.46.0   reportlab==4.2.5

!pip install -q pytest==8.3.4

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import numpy as np
import pandas as pd
import cv2
import timm
import torch
print("ready")

## 3. Google Drive + Kaggle Credentials

Use Drive for persistent checkpoints and processed images. Upload your `kaggle.json` from Kaggle Account settings.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, os

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/aptos_dr")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Upload kaggle.json when prompted.
if not Path("/root/.kaggle/kaggle.json").exists():
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
        shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

## 4. Download APTOS 2019 Dataset

Expected local structure after extraction:

```text
data/raw/aptos2019/train.csv
data/raw/aptos2019/test.csv
data/raw/aptos2019/train_images/*.png
data/raw/aptos2019/test_images/*.png
```

In [ ]:
RAW_DIR = PROJECT_DIR / "data/raw/aptos2019"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if not (RAW_DIR / "train.csv").exists():
    !kaggle competitions download -c aptos2019-blindness-detection -p data/raw/aptos2019
    !unzip -q data/raw/aptos2019/aptos2019-blindness-detection.zip -d data/raw/aptos2019

!find data/raw/aptos2019 -maxdepth 2 -type f | head

## 5. Phase 1: EDA And Quality Audit

This creates class distribution plots, image-size distribution, samples per class, and quality CSVs for black/low-contrast/duplicate images.

In [ ]:
!python scripts/run_eda.py --config configs/config.yaml

from IPython.display import display, Image
for p in ["reports/figures/class_distribution.png", "reports/figures/image_size_distribution.png", "reports/figures/samples_per_class.png"]:
    print(p)
    display(Image(filename=p))

quality = pd.read_csv("reports/quality/image_quality_summary.csv")
display(quality)

### EDA Mistakes To Avoid

- Do not use random train/validation splits.
- Do not delete suspicious images blindly; inspect and document.
- Do not balance or augment validation data.
- Do not optimize anything on the test set.

## 6. Phase 2: Preprocessing

Default: Ben Graham preprocessing at 512px for T4 training. Later, try 768px for final runs.

512px: faster, good for iteration. 768px: better lesion detail, slower, smaller batch.

In [ ]:
# For first T4 run, use 512. Later repeat with --image-size 768.
!python scripts/preprocess_images.py --split train --method ben_graham --image-size 512
!python scripts/preprocess_images.py --split test --method ben_graham --image-size 512

# Optional alternative to compare:
# !python scripts/preprocess_images.py --split train --method clahe --image-size 512

## 7. Phase 3: Stratified 5-Fold Splits

APTOS is small and imbalanced. Use Stratified K-Fold, never a random split.

In [ ]:
!python scripts/create_folds.py --config configs/config.yaml --output data/interim/aptos2019_folds.csv
folds = pd.read_csv("data/interim/aptos2019_folds.csv")
display(pd.crosstab(folds.fold, folds.diagnosis))

## 8. Phase 4: Model Strategy

Run in this order:

1. **Approach A**: EfficientNet-B4 classifier. Fast sanity baseline.
2. **Approach B**: EfficientNet-B4 regression. Usually better aligned with QWK.
3. **Approach C**: EfficientNet-B5/B6 or ConvNeXt with ordinal thresholds + GeM. Best final candidate.

For a single free T4, start with regression B4 at 512px. A realistic first target is validation QWK 0.82-0.88. Pushing 0.90+ usually needs stronger preprocessing, threshold search, 5-fold ensembling, maybe external pretraining, and careful cleanup.

In [ ]:
# Patch config for a T4-friendly first run: regression EfficientNet-B4, 512px, small batch.
import yaml
from pathlib import Path

cfg_path = Path("configs/config.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["preprocessing"]["image_size"] = 512
cfg["model"]["architecture"] = "tf_efficientnet_b4_ns"
cfg["model"]["task"] = "regression"
cfg["model"]["dropout"] = 0.3
cfg["training"]["batch_size"] = 8
cfg["training"]["epochs"] = 10  # increase to 20-30 for real training
cfg["training"]["num_workers"] = 2
cfg["training"]["lr"] = 2e-4
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

## 9. Phase 5: Train One Fold

Use one fold first to validate the whole stack. Turn on W&B if you have an account; otherwise use `--no-wandb`.

In [ ]:
# Smoke/full first fold training. Remove --no-wandb after wandb login if desired.
!python scripts/train.py   --config configs/config.yaml   --fold-csv data/interim/aptos2019_folds.csv   --fold 0   --processed-dir data/processed/aptos2019/train/512px/ben_graham   --use-weighted-sampler   --no-wandb

## 10. Train All 5 Folds

This is the first serious ensemble. On free Colab, save checkpoints to Drive after each fold because sessions can disconnect.

In [ ]:
for fold in range(5):
    print("TRAIN FOLD", fold)
    !python scripts/train.py       --config configs/config.yaml       --fold-csv data/interim/aptos2019_folds.csv       --fold {fold}       --processed-dir data/processed/aptos2019/train/512px/ben_graham       --use-weighted-sampler       --no-wandb
    !cp -v models/checkpoints/*.pth /content/drive/MyDrive/aptos_dr/

## 11. Threshold Optimization

For regression and ordinal models, do not rely only on `round()`. Use validation predictions to optimize four thresholds for QWK.

The repo already provides `optimize_thresholds`; once validation prediction collection is added for every checkpoint, optimize thresholds on out-of-fold predictions only.

In [ ]:
from dr_grading.inference.thresholds import optimize_thresholds
from dr_grading.training.metrics import regression_to_classes, quadratic_weighted_kappa

# Example only. Replace with real OOF continuous predictions from all folds.
y_true = np.array([0,0,1,1,2,2,3,3,4,4])
y_pred = np.array([0.1,0.2,0.9,1.2,2.0,2.1,3.0,3.1,3.8,4.0])
th = optimize_thresholds(y_true, y_pred)
print("thresholds", th)
print("qwk", quadratic_weighted_kappa(y_true, regression_to_classes(y_pred, th)))

## 12. Phase 6: Inference And Submission

Use TTA and average continuous predictions from all fold checkpoints. Do not average rounded labels.

In [ ]:
# Example for one checkpoint. Repeat --checkpoint/--architecture/--task for every fold/model.
# Update checkpoint path names to match your actual saved files.
!python scripts/predict_submission.py   --test-csv data/raw/aptos2019/test.csv   --test-image-dir data/raw/aptos2019/test_images   --checkpoint models/checkpoints/tf_efficientnet_b4_ns_fold0.pth   --architecture tf_efficientnet_b4_ns   --task regression   --thresholds 0.5 1.5 2.5 3.5   --image-size 512   --output submission.csv

!head submission.csv

## 13. Phase 7: Explainable AI Overview

XAI is a clinical trust module, not a visualization afterthought. The goal is to verify that the model is attending to plausible diabetic retinopathy evidence: microaneurysms, hemorrhages, hard exudates, cotton-wool spots, neovascularization, vessel changes, and macular involvement.

Use XAI on these groups:

- true high-confidence examples from each grade
- false positives and false negatives
- adjacent-grade disagreements, especially grade 2 vs 3
- images with artifacts, poor illumination, borders, or camera marks
- cases where thresholding changes the predicted grade

Rule: a good QWK score does not prove clinical reliability. Heatmaps that focus on black borders, camera artifacts, or image corners are a serious warning.

### 13A. Grad-CAM And Variants

The repo currently includes the initial Grad-CAM wrapper in `src/explainability/gradcam.py`. The full production module should support:

- GradCAM: baseline gradient localization
- GradCAM++: better for multiple lesion-like regions
- EigenCAM: gradient-free PCA over activations, stable and fast
- ScoreCAM: perturbation-based, visually strong but slower
- LayerCAM: finer local detail
- AblationCAM: more faithful but expensive

Target layer selection matters because timm internals differ. For this project:

- EfficientNet B4/B5/B6: use the final block or final features layer
- EfficientNetV2: use final block/features layer
- ConvNeXt: use final stage block

The repo utility `get_target_layer(model, arch_name)` handles these cases.

In [ ]:
# Example: generate a Grad-CAM overlay from one trained checkpoint.
# Fill in an existing checkpoint and image id after training.

from pathlib import Path
import numpy as np
import torch
from PIL import Image as PILImage
from IPython.display import display

from dr_grading.inference.predictor import CheckpointSpec, load_checkpoint_model
from dr_grading.data.preprocessing import PreprocessOptions, preprocess_fundus_image
from dr_grading.data.transforms import build_valid_transforms
from explainability.gradcam import GradCAMExplainer

# checkpoint_path = Path("models/checkpoints/tf_efficientnet_b4_ns_fold0.pth")
# image_path = next(Path("data/raw/aptos2019/test_images").glob("*.png"))
# spec = CheckpointSpec(path=checkpoint_path, architecture="tf_efficientnet_b4_ns", task="regression")
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = load_checkpoint_model(spec, device)
# original = np.asarray(PILImage.open(image_path).convert("RGB"))
# processed = preprocess_fundus_image(original, PreprocessOptions(image_size=512), method="ben_graham")
# tensor = build_valid_transforms(512)(image=processed)["image"]
# explainer = GradCAMExplainer(model, arch_name=spec.architecture, device=device)
# cam_result = explainer.explain(
#     image_tensor=tensor,
#     original_image=original,
#     preprocessed_image=processed,
#     regression=True,
#     opacity=0.35,
# )
# display(cam_result.overlay_original)
# display(cam_result.overlay_preprocessed)

### 13B. Multi-Class CAM Grid

For classifier or ordinal models, generate class-specific CAMs for all grades 0-4. The grid answers: "what evidence would push this image toward grade N?"

This is especially useful for adjacent grades:

- grade 1 vs 2: early lesions vs moderate lesion burden
- grade 2 vs 3: moderate vs severe hemorrhage/exudate burden
- grade 3 vs 4: severe NPDR vs proliferative DR evidence

For pure one-output regression models, use a regression target that differentiates the scalar output with respect to the target feature map. The repo includes `RegressionOutputTarget` for this compatibility layer.

In [ ]:
# Skeleton for a 1x5 CAM grid. Use for classification/ordinal checkpoints.
# For regression models, use regression=True and inspect scalar-output attention.

import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 5, figsize=(20, 4))
# for grade in range(5):
#     result = explainer.explain(
#         image_tensor=tensor,
#         original_image=original,
#         preprocessed_image=processed,
#         target_category=grade,
#         regression=False,
#     )
#     axes[grade].imshow(result.overlay_preprocessed)
#     axes[grade].set_title(f"Grade {grade}")
#     axes[grade].axis("off")
# plt.tight_layout()

### 13C. LIME

LIME is model-agnostic. It perturbs superpixels and learns a local surrogate explanation. For retinal images:

- use SLIC superpixels
- show top positive superpixels in green
- show negative/contradicting superpixels in red
- use enough samples, usually 1000+, for stable visualizations

LIME is slower than Grad-CAM and can be sensitive to segmentation quality. Use it as a second opinion, not the main production explanation.

In [ ]:
# LIME implementation sketch for Colab experimentation.
# This is notebook-level code; production code should live in src/explainability/lime_explainer.py.

# from lime import lime_image
# from skimage.segmentation import slic, mark_boundaries
# import numpy as np
#
# def explain_with_lime_colab(model, image_np, preprocess_fn, predict_fn, num_samples=1000, top_k=8):
#     explainer = lime_image.LimeImageExplainer()
#     explanation = explainer.explain_instance(
#         image_np,
#         classifier_fn=predict_fn,
#         segmentation_fn=lambda x: slic(x, n_segments=120, compactness=10, sigma=1),
#         top_labels=5,
#         hide_color=0,
#         num_samples=num_samples,
#     )
#     pred_label = explanation.top_labels[0]
#     image, mask = explanation.get_image_and_mask(
#         pred_label,
#         positive_only=False,
#         num_features=top_k,
#         hide_rest=False,
#     )
#     return mark_boundaries(image / 255.0, mask)

### 13D. SHAP DeepExplainer

SHAP estimates feature contribution. For CNN images it is expensive, so use it sparingly:

- background set: sample around 50 images per class if memory allows
- compute on a small validation subset first
- aggregate pixel-level SHAP into an 8x8 grid for global regional importance

Use SHAP when you need deeper audit evidence. Use Grad-CAM for routine per-prediction explanations.

In [ ]:
# SHAP skeleton. Expect this to be slow on T4; start with tiny batches.

# import shap
#
# background_images = []
# for batch in valid_loader:
#     background_images.append(batch["image"])
#     if sum(x.size(0) for x in background_images) >= 50:
#         break
# background = torch.cat(background_images, dim=0)[:50].to(device)
# explainer = shap.DeepExplainer(model, background)
# sample_batch = next(iter(valid_loader))["image"][:4].to(device)
# shap_values = explainer.shap_values(sample_batch)
# shap.image_plot(shap_values, sample_batch.detach().cpu().numpy().transpose(0, 2, 3, 1))

### 13E. Attention Rollout For ViT Branches

If you later add a ViT or hybrid attention model, use attention rollout to visualize patch attention. Compare attention rollout against Grad-CAM:

- agreement between methods increases trust
- disagreement requires manual review
- attention maps alone are not faithful explanations; they show routing, not necessarily causal contribution

In [ ]:
# Attention rollout placeholder for future ViT branch.
# Production implementation should hook attention matrices from transformer blocks.

# def attention_rollout(attentions, discard_ratio=0.0):
#     result = torch.eye(attentions[0].size(-1), device=attentions[0].device)
#     for attention in attentions:
#         heads_fused = attention.mean(dim=1)
#         flat = heads_fused.view(heads_fused.size(0), -1)
#         if discard_ratio > 0:
#             _, indices = flat.topk(int(flat.size(-1) * discard_ratio), largest=False)
#             flat.scatter_(1, indices, 0)
#         identity = torch.eye(heads_fused.size(-1), device=heads_fused.device)
#         heads_fused = (heads_fused + identity) / 2
#         heads_fused = heads_fused / heads_fused.sum(dim=-1, keepdim=True)
#         result = torch.matmul(heads_fused, result)
#     return result

### 13F. Clinical Landmark Overlay

Production XAI should overlay attention with retinal landmarks:

- optic disc: bright circular region, usually off-center
- macula: darker oval near the center/temporal side
- vessel network: Frangi vesselness or morphology-based segmentation

Reliability score idea:

1. Take top 20% CAM activation pixels.
2. Compute percentage inside macula + vessel zones.
3. Penalize attention on background/borders.
4. Warn if score < 0.4.

Important caveat: rule-based landmark detectors are imperfect. Treat the score as a review aid, not ground truth.

In [ ]:
# Notebook-level retinal landmark prototype.
# Production implementation should go into src/explainability/landmarks.py.

import cv2
import numpy as np
from skimage.filters import frangi


def detect_retinal_landmarks_colab(image_np: np.ndarray) -> dict:
    rgb = image_np.astype(np.uint8)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    # Optic disc: threshold bright regions and choose largest plausible component.
    blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=max(h, w) / 60)
    _, bright = cv2.threshold(blurred, int(np.percentile(blurred, 98)), 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(bright, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    optic_box = None
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(c)
        optic_box = (x, y, bw, bh)

    # Macula: approximate central dark oval zone.
    macula_mask = np.zeros_like(gray, dtype=np.uint8)
    cv2.ellipse(macula_mask, (w // 2, h // 2), (w // 10, h // 14), 0, 0, 360, 255, -1)

    # Vessels: Frangi on inverted grayscale.
    vesselness = frangi(1.0 - gray.astype(np.float32) / 255.0)
    vessel_mask = (vesselness > np.percentile(vesselness, 92)).astype(np.uint8) * 255

    return {"optic_disc_box": optic_box, "macula_mask": macula_mask, "vessel_mask": vessel_mask}


def xai_reliability_score_colab(heatmap: np.ndarray, landmarks: dict) -> dict:
    heatmap = heatmap.astype(np.float32)
    top = heatmap >= np.percentile(heatmap, 80)
    clinical = (landmarks["macula_mask"] > 0) | (landmarks["vessel_mask"] > 0)
    if top.sum() == 0:
        return {"score": 0.0, "clinical_pct": 0.0, "background_pct": 1.0}
    clinical_pct = float((top & clinical).sum() / top.sum())
    return {"score": clinical_pct, "clinical_pct": clinical_pct, "background_pct": 1.0 - clinical_pct}

### 13G. Counterfactual Explanations

Counterfactual question: what minimal visual change would move the prediction down or up a grade?

For images, start with a constrained gradient method:

- freeze model
- optimize a small image delta
- constrain delta with L1/L2 and total variation penalties
- stop when target grade threshold is crossed
- visualize the absolute difference map

Do not present counterfactuals as literal clinical edits. Phrase them as model-sensitive regions.

In [ ]:
# Counterfactual skeleton. Use low resolution and strong regularization.

# def gradient_counterfactual(model, image_tensor, target_value, steps=80, lr=0.01, lam_l1=0.01):
#     model.eval()
#     base = image_tensor.detach().clone().unsqueeze(0).to(device)
#     delta = torch.zeros_like(base, requires_grad=True)
#     optimizer = torch.optim.Adam([delta], lr=lr)
#     for _ in range(steps):
#         candidate = (base + delta).clamp(-3, 3)
#         output = model(candidate).reshape(-1)[0]
#         loss = (output - target_value).pow(2) + lam_l1 * delta.abs().mean()
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#     return delta.detach().cpu()[0]

### 13H. XAI Faithfulness Metrics

Pretty heatmaps are not enough. Measure explanation quality:

- Deletion AUC: remove most-important pixels and measure prediction drop. Lower is better.
- Insertion AUC: start blurred and reveal important pixels. Higher is better.
- Pointing game: heatmap peak should fall inside pathology annotation if available.
- Stability: small image noise should not wildly change the heatmap.

Use these metrics to compare GradCAM, GradCAM++, ScoreCAM, EigenCAM, and AblationCAM.

In [ ]:
# Insertion/deletion AUC skeleton for one image.

# from sklearn.metrics import auc
#
# def compute_insertion_deletion_auc_colab(model, image_tensor, heatmap, steps=20):
#     flat_order = np.argsort(heatmap.reshape(-1))[::-1]
#     _, h, w = image_tensor.shape
#     original = image_tensor.clone()
#     blurred = torch.nn.functional.avg_pool2d(original.unsqueeze(0), 15, stride=1, padding=7)[0]
#     deletion_scores, insertion_scores = [], []
#     for i in range(steps + 1):
#         k = int(len(flat_order) * i / steps)
#         mask = torch.zeros(h * w, dtype=torch.bool)
#         mask[flat_order[:k]] = True
#         mask = mask.view(h, w)
#         deleted = original.clone(); deleted[:, mask] = 0
#         inserted = blurred.clone(); inserted[:, mask] = original[:, mask]
#         with torch.no_grad():
#             deletion_scores.append(float(model(deleted.unsqueeze(0).to(device)).reshape(-1)[0].cpu()))
#             insertion_scores.append(float(model(inserted.unsqueeze(0).to(device)).reshape(-1)[0].cpu()))
#     x = np.linspace(0, 1, steps + 1)
#     return {"deletion_auc": auc(x, deletion_scores), "insertion_auc": auc(x, insertion_scores)}

## 14. Phase 8: Production FastAPI

The local repo should serve the trained ensemble. In production, the API should:

- accept file upload or base64 image
- validate corrupt/invalid images
- apply the exact training preprocessing
- run model ensemble + thresholds
- return grade, severity label, confidence, and Grad-CAM overlay
- expose `/health`
- expose `/explain` and `/explain/methods` once Phase 7 is complete

Use Colab only to train/export checkpoints. Run the API locally or on a server with CPU/GPU depending on latency requirements.

In [ ]:
# Minimal FastAPI smoke command after Phase 8 code exists locally:
# !uvicorn api.main:app --host 0.0.0.0 --port 8000
#
# In Colab, expose only for demos:
# !pip install -q pyngrok
# from pyngrok import ngrok
# public_url = ngrok.connect(8000)
# print(public_url)

## 15. Phase 9: Streamlit Demo App

The Streamlit app should be a clinician-facing demo, not a landing page. Required views:

- upload fundus image
- raw vs Ben Graham preprocessed image
- predicted grade and confidence
- probability/score bar chart
- Grad-CAM overlay
- XAI tab with method dropdown, opacity slider, landmark toggle, reliability score, clinical flags, and PDF export

Do not overstate clinical certainty. Include model limitations and manual-review messaging for low-confidence or low-reliability cases.

In [ ]:
# Streamlit demo command after Phase 9 code exists locally:
# !streamlit run streamlit_app/app.py --server.port 8501 --server.address 0.0.0.0

## 16. Phase 10: Dockerization And Deployment

Production deployment artifacts should include:

- FastAPI Dockerfile, preferably slim Python base
- docker-compose with API + Streamlit
- model weight download script from HuggingFace Hub, GCS, or Drive export
- GitHub Actions workflow that runs pytest and import checks
- pinned CPU/GPU dependency variants

For CPU inference, use smaller backbones or ONNX/TorchScript optimization. For GPU inference, batch requests if throughput matters.

In [ ]:
# Local deployment commands after Phase 10 code exists:
# !docker compose config
# !docker compose build
# !docker compose up

## 17. Optional Kaggle 2015 Pretraining

Use Kaggle 2015 for pretraining or warm-starting, not for APTOS validation. The validation folds and threshold search should remain APTOS-only.

Recommended strategy:

1. Train on 2015 at 512px with noisy labels for a few epochs.
2. Save backbone checkpoint.
3. Fine-tune on APTOS 5 folds.
4. Optimize thresholds on APTOS OOF predictions.

Before mixing datasets, normalize labels, inspect image format differences, and keep a `source` column for auditing.

## 18. Production Export Checklist

After training:

- Copy checkpoints to Drive.
- Save `thresholds.json` from OOF optimization.
- Save config YAML used for the run.
- Save fold CSV used for validation.
- Generate `submission.csv`.
- Run XAI review on representative validation cases.
- Export best checkpoints into `models/checkpoints/` locally.
- Wire the same preprocessing into API and Streamlit.

Most common production mistakes:

- training with one preprocessing method and serving with another
- optimizing thresholds on test predictions
- averaging rounded labels instead of continuous predictions
- using raw accuracy as a model-selection metric
- letting external data leak into validation
- trusting Grad-CAM screenshots without faithfulness checks

## 19. Suggested T4 Run Plan

Use this progression:

1. 512px EfficientNet-B4 regression, fold 0 only, 3-5 epochs: verify stack.
2. 512px EfficientNet-B4 regression, 5 folds, 20-30 epochs.
3. Optimize OOF thresholds.
4. 512px EfficientNet-B5 ordinal/GeM, 5 folds.
5. Optional 768px fine-tune if runtime/VRAM allows.
6. Ensemble B4 regression + B5 ordinal.
7. Run XAI review on high-confidence correct, low-confidence, and severe-grade samples.

A free T4 can get you a strong research model, but QWK 0.92+ is aggressive. Expect iteration on folds, thresholds, preprocessing, external pretraining, ensemble weights, and error analysis.